In [ ]:
import sqlite3
import random
import string

SEED = 42
random.seed(SEED)

db_name = input("Enter SQLite DB filename: ").strip()
conn = sqlite3.connect(db_name)
cur = conn.cursor()

def run_sql(sql: str):
    try:
        cur = conn.execute(sql)
        if sql.lstrip().upper().startswith(("SELECT", "PRAGMA")):
            rows = cur.fetchall()
            return rows
        else:
            conn.commit()
            print("OK")
            return None

    except sqlite3.Error as err:
        print(f"SQLite error: {err}")
        return None

## Creating Table Schema

In [ ]:
try:
    cur.execute("PRAGMA foreign_keys = ON;")

    #Drop existing tables
    cur.executescript("""
        DROP TABLE IF EXISTS ISA;
        DROP TABLE IF EXISTS HAS;
        DROP TABLE IF EXISTS People;
        DROP TABLE IF EXISTS Office;
        DROP TABLE IF EXISTS Room;
        DROP TABLE IF EXISTS Building;
        """)

    #Create tables
    cur.executescript("""
        CREATE TABLE Building (
            name TEXT PRIMARY KEY,
            location TEXT,
            numRooms INT,
            accessibility BOOL,
            capacity INT
        );

        CREATE TABLE Room (
            number TEXT PRIMARY KEY,
            capacity INT,
            sqft INT,
            accessibility BOOL,
            building TEXT,
            FOREIGN KEY (building) REFERENCES Building(name)
        );

        CREATE TABLE Office (
            location TEXT PRIMARY KEY,
            department TEXT
        );

        CREATE TABLE People (
            id INT PRIMARY KEY,
            name TEXT,
            student BOOL,
            faculty BOOL,
            staff BOOL,
            department TEXT
        );

        CREATE TABLE HAS (
            id INT PRIMARY KEY,
            location TEXT,
            name TEXT,
            department TEXT,
            FOREIGN KEY (location) REFERENCES Office(location)
        );

        CREATE TABLE ISA (
            location TEXT PRIMARY KEY,
            roomNumber TEXT,
            FOREIGN KEY (location) REFERENCES Office(location),
            FOREIGN KEY (roomNumber) REFERENCES Room(number)
        );
        """)
    
except Exception as e:
    conn.rollback()
    print("Something went wrong, rolling back.")
    print(e)

# Populating The Database

In [5]:

def rand_text(prefix="", length=8):
    return prefix + ''.join(random.choices(string.ascii_letters + string.digits, k=length))

def rand_bool():
    return random.choice([0, 1])

def main():
    
    #Create Buildings 
    buildings = []
    for i in range(100):
        name = f"B{i}_{rand_text('', 5)}"
        buildings.append(name)
        cur.execute("""
            INSERT INTO Building (name, location, numRooms, accessibility, capacity)
            VALUES (?, ?, ?, ?, ?)
        """, (
            name,
            random.choice([rand_text("Loc_", 6)]), 
            random.randint(1, 500),  
            rand_bool(),
            random.randint(0, 10000)
        ))

    #Create Rooms  
    rooms = []
    for i in range(1000):
        number = f"R{i}_{rand_text('', 4)}"
        rooms.append(number)
        cur.execute("""
            INSERT INTO Room (number, capacity, sqft, accessibility, building)
            VALUES (?, ?, ?, ?, ?)
        """, (
            number,
            random.randint(0, 500),
            random.randint(500, 10000),  
            rand_bool(),
            random.choice(buildings)
        ))

    #Create Offices  
    offices = []
    for i in range(1000):
        loc = f"O{i}_{rand_text('', 4)}"
        offices.append(loc)
        cur.execute("""
            INSERT INTO Office (location, department)
            VALUES (?, ?)
        """, (
            loc,
            random.choice([
                "CS", "Math", "History", "Physics",
                "", None, rand_text("Dept_", 3)
            ])
        ))

    #Create People  
    for i in range(1000):
        cur.execute("""
            INSERT INTO People (id, name, student, faculty, staff, department)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (
            i,
            random.choice([
            rand_text("Name_", 5),
                ""]),
            rand_bool(),
            rand_bool(),
            rand_bool(),
            random.choice(["CS", "Math","History", "Physics", "", rand_text("Dept_", 2)])
        ))

    #Create HAS  
    for i in range(1000):
        cur.execute("""
            INSERT INTO HAS (id, location, name, department)
            VALUES (?, ?, ?, ?)
        """, (
            i,
            random.choice(offices),
            random.choice([
                rand_text("Obj_", 5),
                None,
                ""
            ]),
            random.choice(["CS", None, "", rand_text("Dept_", 2)])
        ))

    #Create ISA  
    #One-to-one with Office due to PK on location
    for loc in offices:
        cur.execute("""
            INSERT INTO ISA (location, roomNumber)
            VALUES (?, ?)
        """, (
            loc,
            random.choice(rooms)
        ))

    conn.commit()
    print("Database populated successfully. Ship it.")


if __name__ == "__main__":
    main()

Database populated successfully. Ship it.


## Question 1: 
# 

In [ ]:
#Write code here